<a href="https://colab.research.google.com/github/Nquyen264-2005/CNNET/blob/main/Do_An_DeepLearning4_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# SentiViet Pro – BiLSTM + Attention (Gradio / Google Colab)
# ============================================================
# ✅ PHIÊN BẢN CẢI THIỆN LOSS / GIẢM OVERFITTING
#
# DANH SÁCH LỖI ĐÃ FIX:
#  [1] Giảm độ phức tạp mô hình: 6.8M params → ~2.4M params
#      (Model cũ có ~834 params/sample → quá dễ học thuộc lòng dữ liệu)
#  [2] Giảm MAX_SEQ_LENGTH: 120 → 60
#      (Văn bản TB chỉ 15 từ, max 43 → không cần padding 120)
#  [3] Sửa MIN_EPOCHS: 100 → 30
#      (Buộc train 100 epoch dù đã overfit từ epoch ~40 → loss val tăng mãi)
#  [4] Sửa PATIENCE: 45 → 15
#      (45 epoch chờ đợi + min 100 epoch = overfit nghiêm trọng)
#  [5] Tăng Dropout: SpatialDropout 0.25→0.35, Dense 0.4→0.5, 0.3→0.4
#  [6] Thêm L2 regularization vào lớp Dense (kernel_regularizer)
#  [7] Tăng label_smoothing: 0.05 → 0.10
#  [8] Sửa data leakage: Tokenizer chỉ fit trên train (không fit val)
#  [9] Đơn giản hoá kiến trúc:
#      - Bỏ lớp BiLSTM thứ 2 (không cần thiết)
#      - Bỏ residual connection cnn_proj + lstm (gây phức tạp vô ích)
#      - Giảm Dense head: 512→256, bỏ lớp 128
#  [10] Giảm batch_size: 128 → 64 (gradient đa dạng hơn, tổng quát hoá tốt hơn)
# ============================================================

import numpy as np
import pandas as pd
import re
import io
import traceback
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Layer, Dropout, SpatialDropout1D,
    GlobalMaxPooling1D, GlobalAveragePooling1D,
    Conv1D, concatenate, LayerNormalization,
    BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, LambdaCallback, Callback
)

# ============================================================
# CỐ ĐỊNH RANDOM SEED ĐỂ KẾT QUẢ ỔN ĐỊNH HƠN GIỮA CÁC LẦN TRAIN
# ============================================================
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================
# TỰ ĐỘNG CẤU HÌNH GPU
# ============================================================
gpus = tf.config.list_physical_devices('GPU')
GPU_STATUS = "⚠️ Đang chạy trên CPU (Không tìm thấy GPU)"
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        try:
            tf.keras.mixed_precision.set_global_policy('mixed_float16')
        except AttributeError:
            tf.keras.mixed_precision.experimental.set_policy('mixed_float16')
        GPU_STATUS = f"✅ GPU: {len(gpus)} thiết bị (Mixed Float16 bật)"
    except RuntimeError as e:
        GPU_STATUS = f"⚠️ Lỗi GPU: {e}"

print(GPU_STATUS)

# ============================================================
# TRẠNG THÁI TOÀN CỤC
# ============================================================
STATE = {
    "texts":        [],
    "labels":       [],
    "val_texts":    [],
    "val_labels":   [],
    "model":        None,
    "tokenizer":    None,
    "history":      None,
    "epoch_log":    [],
    "model_trained": False,
    "comparison_results": None,
    "comparison_reports": "",
}

# ============================================================
# 1. TIỀN XỬ LÝ VĂN BẢN
# ============================================================
TEEN_DICT = {
    "ko": "không", "k": "không", "kh": "không", "hok": "không",
    "dc": "được",  "đc": "được",  "ok": "được",
    "mk": "mình",  "mn": "mọi người", "vs": "với",
    "bth": "bình thường", "bt": "bình thường",
    "nx": "nữa",   "th": "thì",   "vc": "việc",
    "ns": "nói",   "ngta": "người ta", "cx": "cũng",
    "tốt": "tốt",  "hay": "hay",  "chán": "chán",
}

def normalize_teen_code(text: str) -> str:
    words = text.split()
    return " ".join(TEEN_DICT.get(w, w) for w in words)

def preprocess_text(text: str) -> str:
    text = text.lower().strip()
    text = normalize_teen_code(text)
    text = re.sub(
        r'[^\w\sàáâãèéêìíòóôõùúýăđơưạảấầẩẫậắằẳẵặẹẻẽếềểễệỉịọỏốồổỗộớờởỡợụủứừửữựỳỵỷỹ0-9]',
        ' ', text
    )
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ============================================================
# 2. DATA AUGMENTATION NHẸ
# ============================================================
def augment_text(text: str, p: float = 0.15) -> str:
    words = text.split()
    if len(words) > 3 and np.random.rand() < p:
        mid = words[1:-1]
        np.random.shuffle(mid)
        words = [words[0]] + mid + [words[-1]]
    return " ".join(words)

def augment_dataset(texts, labels, factor: int = 2):
    aug_texts, aug_labels = list(texts), list(labels)
    for t, l in zip(texts, labels):
        for _ in range(factor - 1):
            aug_texts.append(augment_text(t))
            aug_labels.append(l)
    return aug_texts, aug_labels

# ============================================================
# 3. CUSTOM CALLBACK
# ============================================================
class MinEpochEarlyStopping(Callback):
    """EarlyStopping có min_epochs và hỗ trợ mode='min' hoặc 'max'.
    Dùng mode='min' cho val_loss sẽ giúp giảm overfitting rõ hơn so với val_accuracy.
    """
    def __init__(self, monitor='val_loss', patience=7,
                 min_epochs=8, restore_best_weights=True,
                 min_delta=2e-3, mode='min'):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.min_epochs = min_epochs
        self.restore_best_weights = restore_best_weights
        self.min_delta = min_delta
        self.mode = mode
        self.wait = 0
        self.best_weights = None
        self.stopped_epoch = 0

        if mode == 'min':
            self.best = np.inf
        else:
            self.best = -np.inf

    def _is_improvement(self, current):
        if self.mode == 'min':
            return current < self.best - self.min_delta
        return current > self.best + self.min_delta

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            return

        if self._is_improvement(current):
            self.best = current
            self.wait = 0
            if self.restore_best_weights:
                self.best_weights = self.model.get_weights()
        else:
            self.wait += 1

        if epoch + 1 >= self.min_epochs and self.wait >= self.patience:
            self.stopped_epoch = epoch
            self.model.stop_training = True
            if self.restore_best_weights and self.best_weights is not None:
                self.model.set_weights(self.best_weights)
            print(f"⛔ Dừng sớm tại epoch {epoch+1} | Best {self.monitor}: {self.best:.4f}")

# ============================================================
# 4. LỚP ATTENTION TUỲ BIẾN
# ============================================================
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        d = input_shape[-1]
        # [FIX 9] Giảm attention hidden dim: 128 → 64
        self.W = self.add_weight(name='attn_W', shape=(d, 48),
                                 initializer='glorot_uniform', trainable=True)
        self.U = self.add_weight(name='attn_U', shape=(48, 1),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='attn_b', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)
        self.scale = float(d) ** 0.5
        super().build(input_shape)

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) / self.scale)
        e = K.dot(e, self.U) + self.b
        a = K.softmax(e, axis=1)
        return K.sum(x * a, axis=1)

# ============================================================
# 5. SIÊU THAM SỐ  ← CÁC THAY ĐỔI CHÍNH
# ============================================================
MAX_VOCAB_SIZE = 15000        # đủ dùng cho dataset hiện tại
MAX_SEQ_LENGTH = 60           # 99% câu <= ~31 từ, max khoảng 48 từ → 60 là an toàn
EMBEDDING_DIM  = 96           # giảm nhẹ để bớt học thuộc
LSTM_UNITS     = 64           # giảm từ 128 → 64 để chống overfit
CNN_FILTERS    = 48           # giảm từ 64 → 48
KERNEL_SIZE    = 3
MIN_EPOCHS     = 8            # không ép train quá lâu khi val_loss đã xấu đi
MAX_EPOCHS     = 80           # đủ cho dữ liệu 8K câu
PATIENCE       = 7            # dừng sớm nhanh hơn

# ============================================================
# 6. XÂY DỰNG MÔ HÌNH  ← ĐÃ ĐƠN GIẢN HOÁ
# ============================================================
def build_model(vocab_size: int) -> Model:
    inputs = Input(shape=(MAX_SEQ_LENGTH,), name='Input')

    # --- Embedding ---
    x = Embedding(vocab_size, EMBEDDING_DIM, name='Embedding')(inputs)
    x = SpatialDropout1D(0.40)(x)   # tăng nhẹ để giảm overfit

    # --- CNN Multi-scale ---
    cnn3 = Conv1D(CNN_FILTERS, 3, activation='relu', padding='same')(x)
    cnn4 = Conv1D(CNN_FILTERS, 4, activation='relu', padding='same')(x)
    cnn5 = Conv1D(CNN_FILTERS, 5, activation='relu', padding='same')(x)
    cnn_out = concatenate([cnn3, cnn4, cnn5])       # shape: (seq, CNN_FILTERS*3=192)
    cnn_out = BatchNormalization()(cnn_out)

    # --- BiLSTM (1 lớp là đủ cho bài toán này) ---
    # [FIX 9] Bỏ lớp BiLSTM thứ 2 - mô hình cũ quá sâu cho ~8K mẫu
    lstm_out = Bidirectional(
        LSTM(LSTM_UNITS, return_sequences=True,
             dropout=0.35, recurrent_dropout=0)   # tăng nhẹ dropout
    )(cnn_out)                                     # shape: (seq, LSTM_UNITS*2=256)
    lstm_out = LayerNormalization()(lstm_out)

    # [FIX 9] BỎ residual connection cnn_proj + merged
    # (residual cần cùng dimension; áp đặt shape khiến mô hình phức tạp vô ích)

    # --- Attention + Pooling ---
    attn_out = AttentionLayer()(lstm_out)          # shape: (LSTM_UNITS*2,)
    max_pool = GlobalMaxPooling1D()(lstm_out)
    avg_pool = GlobalAveragePooling1D()(lstm_out)
    concat   = concatenate([attn_out, max_pool, avg_pool])  # (768,)

    # --- Dense Head (đơn giản hơn) ---
    # [FIX 9] 512→256, bỏ lớp Dense(128)
    # [FIX 6] Thêm L2 regularization để penalise trọng số lớn
    x = Dense(128, activation='relu',
              kernel_regularizer=l2(2e-4))(concat)
    x = LayerNormalization()(x)
    x = Dropout(0.55)(x)   # tăng để chống học thuộc

    x = Dense(64, activation='relu',
              kernel_regularizer=l2(2e-4))(x)
    x = Dropout(0.45)(x)

    outputs = Dense(3, activation='softmax', dtype='float32', name='Output')(x)
    model   = Model(inputs, outputs)

    # FIX: dùng learning_rate dạng số cố định để ReduceLROnPlateau có thể tự giảm LR.
    # Không dùng đồng thời LearningRateSchedule + ReduceLROnPlateau vì dễ lỗi khi callback đổi LR.
    optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4)
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.CategoricalCrossentropy(
            label_smoothing=0.05  # giảm từ 0.10: loss dễ nhìn hơn nhưng vẫn chống quá tự tin
        ),
        metrics=['accuracy']
    )
    return model

# ============================================================

# ============================================================
# 6B. HAM DANH GIA - HIEN THI 4 CHU SO THAP PHAN
# ============================================================
CLASS_LABELS = [0, 1, 2]
CLASS_NAMES_UI = ["Tieu cuc", "Trung tinh", "Tich cuc"]


def format_classification_report(y_true, y_pred):
    """Tao classification report voi 4 chu so thap phan, vi du 0.9893."""
    from sklearn.metrics import classification_report

    return classification_report(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        target_names=CLASS_NAMES_UI,
        digits=4,
        zero_division=0,
    )


def collect_metric_row(model_name, y_true, y_pred, train_time=None, params=None, note=""):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    return {
        "Mo hinh": model_name,
        "Accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "Macro Precision": round(float(macro_p), 4),
        "Macro Recall": round(float(macro_r), 4),
        "Macro F1": round(float(macro_f1), 4),
        "Weighted Precision": round(float(weighted_p), 4),
        "Weighted Recall": round(float(weighted_r), 4),
        "Weighted F1": round(float(weighted_f1), 4),
        "Train time (s)": None if train_time is None else round(float(train_time), 2),
        "Params/Features": "" if params is None else f"{int(params):,}",
        "Ghi chu": note,
    }


def make_comparison_chart(metrics_df):
    if metrics_df is None or len(metrics_df) == 0:
        return None

    fig, ax = plt.subplots(figsize=(11, 5.6))
    metrics = ["Accuracy", "Macro F1", "Weighted F1"]
    colors = ["#38bdf8", "#22c55e", "#f59e0b"]
    x = np.arange(len(metrics_df))
    width = 0.24

    for i, metric in enumerate(metrics):
        values = metrics_df[metric].astype(float).to_numpy()
        bars = ax.bar(x + (i - 1) * width, values, width, label=metric, color=colors[i])
        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                min(value + 0.012, 1.04),
                f"{value:.4f}",
                ha='center',
                va='bottom',
                fontsize=8,
                rotation=90,
            )

    ax.set_title('So sanh truc tiep cac mo hinh', fontsize=13, fontweight='bold')
    ax.set_ylabel('Diem so')
    ax.set_ylim(0, 1.08)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_df["Mo hinh"].tolist(), rotation=18, ha='right')
    ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.0))
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    return fig


def prepare_sequence_train_val():
    tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>', char_level=False, filters='')
    tokenizer.fit_on_texts(STATE['texts'])
    X_train = pad_sequences(tokenizer.texts_to_sequences(STATE['texts']), maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
    X_val = pad_sequences(tokenizer.texts_to_sequences(STATE['val_texts']), maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
    y_train = tf.keras.utils.to_categorical(STATE['labels'], num_classes=3)
    y_val = tf.keras.utils.to_categorical(STATE['val_labels'], num_classes=3)
    vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)
    return X_train, y_train, X_val, y_val, vocab_size


def compile_compare_model(model):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def build_cnn_only_model(vocab_size):
    inputs = Input(shape=(MAX_SEQ_LENGTH,), name='Input')
    x = Embedding(vocab_size, EMBEDDING_DIM, name='Embedding')(inputs)
    x = SpatialDropout1D(0.25)(x)
    x = Conv1D(128, 3, activation='relu', padding='same')(x)
    x = GlobalMaxPooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.35)(x)
    outputs = Dense(3, activation='softmax', dtype='float32', name='Output')(x)
    return compile_compare_model(Model(inputs, outputs, name='CNN_Only'))


def build_lstm_only_model(vocab_size):
    inputs = Input(shape=(MAX_SEQ_LENGTH,), name='Input')
    x = Embedding(vocab_size, EMBEDDING_DIM, name='Embedding')(inputs)
    x = SpatialDropout1D(0.25)(x)
    x = LSTM(96, dropout=0.25, recurrent_dropout=0)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.35)(x)
    outputs = Dense(3, activation='softmax', dtype='float32', name='Output')(x)
    return compile_compare_model(Model(inputs, outputs, name='LSTM_Only'))


def build_bilstm_only_model(vocab_size):
    inputs = Input(shape=(MAX_SEQ_LENGTH,), name='Input')
    x = Embedding(vocab_size, EMBEDDING_DIM, name='Embedding')(inputs)
    x = SpatialDropout1D(0.25)(x)
    x = Bidirectional(LSTM(64, dropout=0.25, recurrent_dropout=0))(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.35)(x)
    outputs = Dense(3, activation='softmax', dtype='float32', name='Output')(x)
    return compile_compare_model(Model(inputs, outputs, name='BiLSTM_Only'))

# 7. ĐỌC DỮ LIỆU BAN ĐẦU
# ============================================================
def load_initial_data():
    label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
    try:
        df_train = pd.read_csv('synthetic_train.csv')
        df_train['label'] = df_train['sentiment'].map(label_map)
        df_train = df_train.dropna(subset=['sentence', 'label'])
        df_train['sentence_clean'] = df_train['sentence'].astype(str).apply(preprocess_text)
        STATE['texts']  = df_train['sentence_clean'].tolist()
        STATE['labels'] = df_train['label'].astype(int).tolist()

        df_val = pd.read_csv('synthetic_val.csv')
        df_val['label'] = df_val['sentiment'].map(label_map)
        df_val = df_val.dropna(subset=['sentence', 'label'])
        df_val['sentence_clean'] = df_val['sentence'].astype(str).apply(preprocess_text)
        STATE['val_texts']  = df_val['sentence_clean'].tolist()
        STATE['val_labels'] = df_val['label'].astype(int).tolist()
        print(f"✅ Đã tải: {len(STATE['texts'])} câu train, {len(STATE['val_texts'])} câu val")

    except Exception as e:
        print(f"⚠️ Không đọc được CSV: {e}. Dùng dữ liệu mẫu.")
        SAMPLE_TRAIN = [
            ("thầy dạy rất hay và dễ hiểu", 2),
            ("cô giảng viên nhiệt tình và tận tâm", 2),
            ("môn học rất bổ ích và thú vị", 2),
            ("tài liệu phong phú dễ tiếp cận", 2),
            ("giờ học sôi nổi và vui", 2),
            ("thầy giải thích rõ ràng chi tiết", 2),
            ("môn học quá chán không muốn học", 0),
            ("thầy giảng khó hiểu quá mệt mỏi", 0),
            ("bài thi khó quá không làm được", 0),
            ("môn này vô ích lãng phí thời gian", 0),
            ("thầy không quan tâm sinh viên", 0),
            ("học mãi không hiểu bực bội ghê", 0),
            ("bình thường thôi không có gì đặc biệt", 1),
            ("cũng được không tệ lắm", 1),
            ("tạm ổn học được", 1),
        ]
        SAMPLE_VAL = [
            ("giảng dạy tốt tôi rất hài lòng", 2),
            ("quá tệ không học được gì cả", 0),
            ("cũng bình thường thôi", 1),
        ]
        STATE['texts']      = [preprocess_text(t) for t, _ in SAMPLE_TRAIN]
        STATE['labels']     = [l for _, l in SAMPLE_TRAIN]
        STATE['val_texts']  = [preprocess_text(t) for t, _ in SAMPLE_VAL]
        STATE['val_labels'] = [l for _, l in SAMPLE_VAL]

load_initial_data()
# ============================================================
# 7B. CHẾ ĐỘ 2: GỘP THÊM 2 FILE CSV VÀO DATASET HIỆN TẠI
#     Lưu ý: phần này CHỈ thêm chức năng load/gộp CSV,
#     KHÔNG thay đổi kiến trúc model, hàm train, hàm predict.
# ============================================================
def strip_accents(text: str) -> str:
    """Bỏ dấu tiếng Việt để nhận diện tên cột / nhãn linh hoạt hơn."""
    import unicodedata
    text = str(text)
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('đ', 'd').replace('Đ', 'D')


def normalize_column_name(col: str) -> str:
    col = strip_accents(str(col)).lower().strip()
    col = re.sub(r'[^a-z0-9]+', '_', col)
    return col.strip('_')


def find_column(df: pd.DataFrame, candidates):
    """Tìm cột theo nhiều tên thường gặp."""
    norm_to_original = {normalize_column_name(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_column_name(cand)
        if key in norm_to_original:
            return norm_to_original[key]
    return None


def read_csv_safely(file_path: str) -> pd.DataFrame:
    """Đọc CSV với nhiều encoding thường gặp trong file tiếng Việt."""
    last_error = None
    for enc in ['utf-8-sig', 'utf-8', 'cp1258', 'latin1']:
        try:
            return pd.read_csv(file_path, encoding=enc)
        except Exception as e:
            last_error = e
    raise ValueError(f"Không đọc được file CSV. Lỗi cuối: {last_error}")


def get_uploaded_path(file_obj):
    """Gradio có thể trả về path, object hoặc dict tuỳ phiên bản."""
    if file_obj is None:
        return None
    if isinstance(file_obj, str):
        return file_obj
    if isinstance(file_obj, dict):
        return file_obj.get('path') or file_obj.get('name')
    if hasattr(file_obj, 'name'):
        return file_obj.name
    return str(file_obj)


def normalize_label_value(value):
    """Chuẩn hoá nhãn về 0: negative, 1: neutral, 2: positive."""
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, np.integer)):
        return int(value) if int(value) in [0, 1, 2] else np.nan
    if isinstance(value, (float, np.floating)):
        return int(value) if value in [0.0, 1.0, 2.0] else np.nan

    raw = str(value).strip().lower()
    raw_no_accent = strip_accents(raw)
    raw_no_accent = re.sub(r'[_\-]+', ' ', raw_no_accent)
    raw_no_accent = re.sub(r'\s+', ' ', raw_no_accent).strip()

    label_map = {
        '0': 0, 'negative': 0, 'neg': 0, 'tieu cuc': 0, 'xau': 0, 'khong hai long': 0,
        '1': 1, 'neutral': 1, 'neu': 1, 'trung tinh': 1, 'binh thuong': 1, 'tam on': 1,
        '2': 2, 'positive': 2, 'pos': 2, 'tich cuc': 2, 'tot': 2, 'hai long': 2,
    }
    return label_map.get(raw_no_accent, np.nan)


def parse_dataset_csv(file_obj, dataset_name='train'):
    """Đọc 1 file CSV và trả về texts, labels, thống kê."""
    file_path = get_uploaded_path(file_obj)
    if not file_path:
        raise ValueError(f"Bạn chưa chọn file {dataset_name}.")

    df = read_csv_safely(file_path)
    if df.empty:
        raise ValueError(f"File {dataset_name} đang rỗng.")

    text_candidates = [
        'sentence', 'text', 'content', 'comment', 'review', 'cau', 'câu',
        'van_ban', 'văn bản', 'noi_dung', 'nội dung', 'document'
    ]
    label_candidates = [
        'sentiment', 'label', 'class', 'target', 'category', 'nhan', 'nhãn',
        'cam_xuc', 'cảm xúc', 'phan_loai', 'phân loại'
    ]

    text_col = find_column(df, text_candidates)
    label_col = find_column(df, label_candidates)

    if text_col is None or label_col is None:
        raise ValueError(
            f"File {dataset_name} cần có cột văn bản và cột nhãn.\n"
            f"Gợi ý định dạng tốt nhất: sentence,sentiment\n"
            f"Các cột hiện có trong file: {list(df.columns)}"
        )

    work = df[[text_col, label_col]].copy()
    work.columns = ['sentence', 'label_raw']
    work['label'] = work['label_raw'].apply(normalize_label_value)
    work['sentence_clean'] = work['sentence'].astype(str).apply(preprocess_text)

    before = len(work)
    work = work.dropna(subset=['sentence_clean', 'label'])
    work = work[work['sentence_clean'].str.len() > 0]
    work['label'] = work['label'].astype(int)
    dropped = before - len(work)

    if len(work) == 0:
        raise ValueError(
            f"File {dataset_name} không còn dòng hợp lệ sau khi xử lý. "
            f"Nhãn hợp lệ: negative/neutral/positive hoặc 0/1/2."
        )

    labels_present = sorted(work['label'].unique().tolist())
    if not set(labels_present).issubset({0, 1, 2}):
        raise ValueError(f"File {dataset_name} có nhãn ngoài phạm vi 0,1,2.")

    return (
        work['sentence_clean'].tolist(),
        work['label'].tolist(),
        {
            'rows_before': before,
            'rows_after': len(work),
            'rows_dropped': dropped,
            'text_col': text_col,
            'label_col': label_col,
            'labels_present': labels_present,
        }
    )


def reset_trained_model_state():
    """Reset model/tokenizer sau khi thay đổi dữ liệu."""
    STATE['model'] = None
    STATE['tokenizer'] = None
    STATE['history'] = None
    STATE['epoch_log'] = []
    STATE['model_trained'] = False
    STATE['comparison_results'] = None
    STATE['comparison_reports'] = ""
    tf.keras.backend.clear_session()


def merge_text_label_lists(old_texts, old_labels, new_texts, new_labels, remove_duplicates=True):
    """Gộp dữ liệu mới vào dữ liệu cũ, có thể bỏ trùng/xung đột nhãn."""
    old_texts = list(old_texts)
    old_labels = [int(x) for x in old_labels]
    new_texts = list(new_texts)
    new_labels = [int(x) for x in new_labels]

    if not remove_duplicates:
        return old_texts + new_texts, old_labels + new_labels, {
            'added': len(new_texts),
            'skipped_duplicate': 0,
            'skipped_conflict': 0,
        }

    seen_by_text = {text: int(label) for text, label in zip(old_texts, old_labels)}
    merged_texts = old_texts[:]
    merged_labels = old_labels[:]
    added = 0
    skipped_duplicate = 0
    skipped_conflict = 0

    for text, label in zip(new_texts, new_labels):
        label = int(label)
        if text in seen_by_text:
            if seen_by_text[text] == label:
                skipped_duplicate += 1
            else:
                skipped_conflict += 1
            continue
        seen_by_text[text] = label
        merged_texts.append(text)
        merged_labels.append(label)
        added += 1

    return merged_texts, merged_labels, {
        'added': added,
        'skipped_duplicate': skipped_duplicate,
        'skipped_conflict': skipped_conflict,
    }


def append_csv_pair(train_file, val_file, remove_duplicates=True):
    """CHẾ ĐỘ 2: Load bắt buộc 2 file CSV và GỘP vào bộ dữ liệu hiện tại."""
    try:
        if train_file is None or val_file is None:
            return (
                "❌ Bạn cần tải đủ 2 file: 1 file Train và 1 file Val/Test để gộp dữ liệu.",
                get_data_info()
            )

        old_train_n = len(STATE['texts'])
        old_val_n = len(STATE['val_texts'])

        train_texts, train_labels, train_info = parse_dataset_csv(train_file, 'train mới')
        val_texts, val_labels, val_info = parse_dataset_csv(val_file, 'val mới')

        merged_train_texts, merged_train_labels, train_merge_info = merge_text_label_lists(
            STATE['texts'], STATE['labels'], train_texts, train_labels,
            remove_duplicates=remove_duplicates
        )
        merged_val_texts, merged_val_labels, val_merge_info = merge_text_label_lists(
            STATE['val_texts'], STATE['val_labels'], val_texts, val_labels,
            remove_duplicates=remove_duplicates
        )

        STATE['texts'] = merged_train_texts
        STATE['labels'] = merged_train_labels
        STATE['val_texts'] = merged_val_texts
        STATE['val_labels'] = merged_val_labels

        reset_trained_model_state()

        duplicate_note = "có bật bỏ trùng" if remove_duplicates else "không bỏ trùng"
        status = (
            f"✅ Chế độ 2: Đã GỘP 2 file CSV vào bộ dữ liệu hiện tại ({duplicate_note}).\n"
            f"📁 Train cũ: {old_train_n} → mới: {len(STATE['texts'])} dòng | thêm thực tế: {train_merge_info['added']} | bỏ trùng: {train_merge_info['skipped_duplicate']} | bỏ xung đột nhãn: {train_merge_info['skipped_conflict']}\n"
            f"📁 Val/Test cũ: {old_val_n} → mới: {len(STATE['val_texts'])} dòng | thêm thực tế: {val_merge_info['added']} | bỏ trùng: {val_merge_info['skipped_duplicate']} | bỏ xung đột nhãn: {val_merge_info['skipped_conflict']}\n"
            f"🧾 File Train mới: đọc {train_info['rows_after']} dòng hợp lệ, bỏ {train_info['rows_dropped']} dòng lỗi. Cột text: `{train_info['text_col']}`, cột nhãn: `{train_info['label_col']}`.\n"
            f"🧾 File Val mới: đọc {val_info['rows_after']} dòng hợp lệ, bỏ {val_info['rows_dropped']} dòng lỗi. Cột text: `{val_info['text_col']}`, cột nhãn: `{val_info['label_col']}`.\n"
            "🔁 Dữ liệu đã thay đổi nên model/tokenizer cũ đã được reset. Hãy train lại từ đầu ở tab Huấn luyện."
        )
        if train_merge_info['skipped_conflict'] > 0 or val_merge_info['skipped_conflict'] > 0:
            status += "\n⚠️ Có câu bị trùng nội dung nhưng khác nhãn. Code đã giữ nhãn cũ để tránh làm nhiễu dữ liệu."

        return status, get_data_info()

    except Exception as e:
        return (
            "❌ Gộp CSV thất bại:\n"
            f"{str(e)}\n\n"
            "Định dạng khuyên dùng:\n"
            "sentence,sentiment\n"
            "thầy dạy rất hay,positive\n"
            "bình thường thôi,neutral\n"
            "môn này quá chán,negative",
            get_data_info()
        )

# ============================================================
# 8. HÀM HUẤN LUYỆN
# ============================================================
def build_and_train_model(progress=gr.Progress()):
    """Huấn luyện mô hình, trả về (status_str, acc_fig, loss_fig, summary_str)"""
    # Xoá graph cũ để tránh đầy RAM/GPU khi bấm train nhiều lần trong Gradio
    tf.keras.backend.clear_session()

    n_raw = len(STATE['texts'])
    if n_raw == 0:
        return "❌ Không có dữ liệu để huấn luyện!", None, None, ""

    # Không augment bằng cách đảo từ nữa, vì câu cảm xúc tiếng Việt phụ thuộc thứ tự từ.
    # Ví dụ: "không tốt" nếu đảo lung tung có thể làm nhiễu nhãn.
    aug_texts, aug_labels = list(STATE['texts']), list(STATE['labels'])
    n_aug = len(aug_texts)
    progress(0, desc=f"Chuẩn bị dữ liệu: {n_raw} câu train…")

    # [FIX 8] Chỉ fit tokenizer trên dữ liệu TRAIN (không fit val → tránh data leakage)
    tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>',
                          char_level=False, filters='')
    tokenizer.fit_on_texts(aug_texts)   # ← đã bỏ STATE['val_texts']

    seqs_train = tokenizer.texts_to_sequences(aug_texts)
    X_train    = pad_sequences(seqs_train, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
    y_train    = tf.keras.utils.to_categorical(aug_labels, num_classes=3)

    seqs_val   = tokenizer.texts_to_sequences(STATE['val_texts'])
    X_val      = pad_sequences(seqs_val, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
    y_val      = tf.keras.utils.to_categorical(STATE['val_labels'], num_classes=3)

    vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

    raw_arr    = np.array(aug_labels, dtype=np.int32)
    counts     = np.bincount(raw_arr).astype(np.float64)
    total      = float(len(raw_arr))
    n_cls      = float(len(np.unique(raw_arr)))
    cw         = total / (n_cls * counts)
    class_weight = {int(i): float(w) for i, w in enumerate(cw)}
    # Dataset hiện tại gần như cân bằng, class_weight chỉ giữ để an toàn khi bạn thêm dữ liệu lệch nhãn.

    model = build_model(vocab_size)

    epoch_log = []
    total_epochs = [0]

    def on_epoch_end_cb(epoch, logs):
        total_epochs[0] = epoch + 1
        entry = {
            'Epoch':         epoch + 1,
            'Train Acc (%)': round(logs.get('accuracy', 0) * 100, 2),
            'Val Acc (%)':   round(logs.get('val_accuracy', 0) * 100, 2),
            'Train Loss':    round(logs.get('loss', 0), 4),
            'Val Loss':      round(logs.get('val_loss', 0), 4),
        }
        epoch_log.append(entry)
        STATE['epoch_log'] = epoch_log
        pct = min((epoch + 1) / MAX_EPOCHS, 1.0)
        progress(pct, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} | Val Acc: {entry['Val Acc (%)']:.2f}%")

    live_cb = LambdaCallback(on_epoch_end=on_epoch_end_cb)

    # Dừng theo val_loss để giảm overfitting và giảm loss thực tế.
    # val_accuracy thường đứng yên quanh 84-85%, trong khi val_loss đã tăng → nếu theo accuracy sẽ train quá lâu.
    min_epoch_es = MinEpochEarlyStopping(
        monitor='val_loss', mode='min', patience=PATIENCE,
        min_epochs=MIN_EPOCHS, restore_best_weights=True, min_delta=2e-3,
    )

    # ReduceLROnPlateau: tự giảm LR khi val_loss không giảm
    # Hoạt động được vì optimizer dùng learning_rate=3e-4 dạng số cố định.
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=64,
        class_weight=class_weight,
        callbacks=[reduce_lr, min_epoch_es, live_cb],
        verbose=1,
        shuffle=True,
    )

    STATE['model']         = model
    STATE['tokenizer']     = tokenizer
    STATE['history']       = history.history
    STATE['model_trained'] = True
    STATE['epoch_log']     = epoch_log

    best_val_acc = max(history.history['val_accuracy']) * 100
    best_val_loss = min(history.history['val_loss'])
    epochs_ran   = len(history.history['val_accuracy'])
    status_msg   = (
        f"✅ Hoàn tất! {epochs_ran} epoch | "
        f"Val Accuracy tốt nhất: {best_val_acc:.2f}% | "
        f"Val Loss tốt nhất: {best_val_loss:.4f} | "
        f"Train: {n_aug} câu | Val: {len(X_val)} câu"
    )

    # -- Vẽ biểu đồ Accuracy --
    fig_acc, ax_acc = plt.subplots(figsize=(7, 4))
    ax_acc.plot(history.history['accuracy'],     label='Train Acc',  color='#6ee7f7', linewidth=2)
    ax_acc.plot(history.history['val_accuracy'], label='Val Acc',    color='#a78bfa', linewidth=2)
    ax_acc.set_title('Accuracy theo Epoch', fontsize=13, fontweight='bold')
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy')
    ax_acc.legend(); ax_acc.grid(alpha=0.3)
    plt.tight_layout()

    # -- Vẽ biểu đồ Loss --
    fig_loss, ax_loss = plt.subplots(figsize=(7, 4))
    ax_loss.plot(history.history['loss'],     label='Train Loss', color='#f87171', linewidth=2)
    ax_loss.plot(history.history['val_loss'], label='Val Loss',   color='#fbbf24', linewidth=2)
    ax_loss.set_title('Loss theo Epoch', fontsize=13, fontweight='bold')
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
    ax_loss.legend(); ax_loss.grid(alpha=0.3)
    plt.tight_layout()

    # -- Model summary --
    summary_lines = []
    model.summary(print_fn=lambda x: summary_lines.append(x))
    summary_str = '\n'.join(summary_lines)

    # -- Bao cao danh gia 4 chu so thap phan --
    y_val_true = np.array(STATE['val_labels'], dtype=np.int32)
    y_val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
    report_str = format_classification_report(y_val_true, y_val_pred)

    return status_msg, fig_acc, fig_loss, summary_str, report_str

# ============================================================
# 9. HÀM DỰ ĐOÁN
# ============================================================
def predict_sentiment(text: str):
    """Trả về (result_html, bar_chart_fig, neg_pct, neu_pct, pos_pct)"""
    if not text.strip():
        return "<p style='color:orange'>⚠️ Vui lòng nhập văn bản!</p>", None, 0, 0, 0

    if not STATE['model_trained']:
        return "<p style='color:red'>❌ Mô hình chưa được huấn luyện! Sang tab <b>Huấn luyện</b> và nhấn nút Huấn luyện.</p>", None, 0, 0, 0

    clean = preprocess_text(text)
    seq   = STATE['tokenizer'].texts_to_sequences([clean])
    pad   = pad_sequences(seq, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
    prob  = np.array(STATE['model'].predict(pad, verbose=0)[0], dtype=np.float64)

    idx        = int(np.argmax(prob))
    label_info = {
        0: ("😡 Tiêu cực", "#ef4444", "#7f1d1d"),
        1: ("😐 Trung tính", "#60a5fa", "#1e3a5f"),
        2: ("😍 Tích cực",  "#10b981", "#064e3b"),
    }
    label_name, border_color, bg_color = label_info[idx]
    conf = prob[idx] * 100

    result_html = f"""
    <div style="
        background:{bg_color}; border:2px solid {border_color};
        border-radius:16px; padding:24px; text-align:center;
        font-family:'Segoe UI',sans-serif; margin:12px 0;
    ">
        <div style="font-size:2rem; font-weight:700; color:{border_color};">{label_name}</div>
        <div style="font-size:1rem; color:#e5e7eb; margin-top:8px;">
            Độ tin cậy: <b>{conf:.2f}%</b>
        </div>
    </div>
    """

    fig, ax = plt.subplots(figsize=(6, 3))
    labels  = ["😡 Tiêu cực", "😐 Trung tính", "😍 Tích cực"]
    colors  = ["#ef4444", "#60a5fa", "#10b981"]
    bars    = ax.barh(labels, prob * 100, color=colors, height=0.5)
    for bar, p in zip(bars, prob * 100):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{p:.1f}%", va='center', fontsize=11, fontweight='bold')
    ax.set_xlim(0, 115)
    ax.set_xlabel('Xác suất (%)')
    ax.set_title('Phân phối xác suất', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()

    return result_html, fig, f"{prob[0]*100:.2f}%", f"{prob[1]*100:.2f}%", f"{prob[2]*100:.2f}%"

# ============================================================
# 10. HÀM THÊM CÂU THỦ CÔNG
# ============================================================
def add_single_sentence(new_text: str, label: int):
    if not new_text.strip():
        return "⚠️ Vui lòng nhập câu văn!", get_data_info()
    clean = preprocess_text(new_text)
    STATE['texts'].append(clean)
    STATE['labels'].append(int(label))
    label_names = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}
    return f"✅ Đã thêm: '{clean}' → {label_names[int(label)]}", get_data_info()

def get_data_info():
    n  = len(STATE['texts'])
    nv = len(STATE['val_texts'])
    if n == 0:
        return "Chưa có dữ liệu."
    from collections import Counter
    c  = Counter(STATE['labels'])
    cv = Counter(STATE['val_labels'])
    lnames = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}
    info  = f"📊 **Train**: {n} câu | "
    info += " | ".join(f"{lnames[k]}: {c[k]}" for k in sorted(c))
    info += f"\n📊 **Val**: {nv} câu | "
    info += " | ".join(f"{lnames[k]}: {cv[k]}" for k in sorted(cv))
    return info


# ============================================================
# ============================================================
# 11. SO SANH MO HINH
# ============================================================
def run_model_comparison(compare_epochs=8, include_phobert=False, progress=gr.Progress()):
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import Pipeline
    except Exception as e:
        return f"Khong import duoc scikit-learn: {e}", pd.DataFrame(), None, ""

    if len(STATE['texts']) == 0 or len(STATE['val_texts']) == 0:
        return "Chua co du lieu train/val de so sanh.", pd.DataFrame(), None, ""

    if not STATE['model_trained'] or STATE['model'] is None or STATE['tokenizer'] is None:
        return "Hay train mo hinh chinh o tab Huan luyen truoc, sau do moi bam so sanh.", pd.DataFrame(), None, ""

    y_true = np.array(STATE['val_labels'], dtype=np.int32)
    results, reports, notes = [], [], []

    progress(0.05, desc="Danh gia mo hinh de xuat...")
    main_X_val = pad_sequences(STATE['tokenizer'].texts_to_sequences(STATE['val_texts']), maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
    main_pred = np.argmax(STATE['model'].predict(main_X_val, verbose=0), axis=1)
    main_epochs = 0 if STATE['history'] is None else len(STATE['history'].get('val_accuracy', []))
    results.append(collect_metric_row("De xuat: CNN-BiLSTM-Attention", y_true, main_pred, params=STATE['model'].count_params(), note=f"Da train truoc ({main_epochs} epoch)"))
    reports.append("===== De xuat: CNN-BiLSTM-Attention =====\n" + format_classification_report(y_true, main_pred))

    progress(0.18, desc="Train Logistic Regression + TF-IDF...")
    start = time.perf_counter()
    tfidf_lr = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1200, class_weight='balanced', C=2.0, solver='lbfgs')),
    ])
    tfidf_lr.fit(STATE['texts'], STATE['labels'])
    pred = tfidf_lr.predict(STATE['val_texts'])
    results.append(collect_metric_row("Logistic Regression + TF-IDF", y_true, pred, train_time=time.perf_counter() - start, params=len(tfidf_lr.named_steps['tfidf'].vocabulary_), note="TF-IDF unigram/bigram"))
    reports.append("===== Logistic Regression + TF-IDF =====\n" + format_classification_report(y_true, pred))

    X_train, y_train, X_val, y_val, vocab_size = prepare_sequence_train_val()
    builders = [("CNN don thuan", build_cnn_only_model, 0.36), ("LSTM", build_lstm_only_model, 0.54), ("BiLSTM", build_bilstm_only_model, 0.72)]
    for model_name, builder, pct in builders:
        progress(pct, desc=f"Train {model_name}...")
        model = builder(vocab_size)
        start = time.perf_counter()
        model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=int(compare_epochs), batch_size=64, callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)], verbose=0, shuffle=True)
        pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
        results.append(collect_metric_row(model_name, y_true, pred, train_time=time.perf_counter() - start, params=model.count_params(), note=f"{int(compare_epochs)} epoch toi da"))
        reports.append(f"===== {model_name} =====\n" + format_classification_report(y_true, pred))

    if include_phobert:
        progress(0.88, desc="Tao embedding PhoBERT...")
        try:
            import torch
            from transformers import AutoModel, AutoTokenizer
            phobert_name = 'vinai/phobert-base'
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            phobert_tokenizer = AutoTokenizer.from_pretrained(phobert_name)
            phobert_model = AutoModel.from_pretrained(phobert_name).to(device)
            phobert_model.eval()

            def encode_phobert(texts, batch_size=16):
                vectors = []
                with torch.no_grad():
                    for start_idx in range(0, len(texts), batch_size):
                        batch_texts = texts[start_idx:start_idx + batch_size]
                        encoded = phobert_tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
                        encoded = {k: v.to(device) for k, v in encoded.items()}
                        output = phobert_model(**encoded)
                        vectors.append(output.last_hidden_state[:, 0, :].detach().cpu().numpy())
                return np.vstack(vectors)

            start = time.perf_counter()
            X_train_bert = encode_phobert(STATE['texts'])
            X_val_bert = encode_phobert(STATE['val_texts'])
            bert_clf = LogisticRegression(max_iter=1200, class_weight='balanced', C=1.0, solver='lbfgs')
            bert_clf.fit(X_train_bert, STATE['labels'])
            pred = bert_clf.predict(X_val_bert)
            results.append(collect_metric_row("Transformer + PhoBERT", y_true, pred, train_time=time.perf_counter() - start, params=phobert_model.num_parameters(), note="PhoBERT embedding + LR"))
            reports.append("===== Transformer + PhoBERT =====\n" + format_classification_report(y_true, pred))
        except Exception as e:
            notes.append("Bo qua Transformer + PhoBERT: " + str(e))
    else:
        notes.append("Chua chay Transformer + PhoBERT. Bat checkbox neu moi truong co transformers/torch va co the tai/cache vinai/phobert-base.")

    metrics_df = pd.DataFrame(results)
    fig = make_comparison_chart(metrics_df)
    reports_text = "\n\n".join(reports)
    STATE['comparison_results'] = metrics_df
    STATE['comparison_reports'] = reports_text
    status = f"Hoan tat so sanh {len(metrics_df)} mo hinh. Cac chi so da hien thi 4 chu so thap phan."
    if notes:
        status += "\n" + "\n".join(notes)
    progress(1.0, desc="Hoan tat so sanh")
    return status, metrics_df, fig, reports_text

# 12. GRADIO UI
# ============================================================
CSS = """
.gradio-container { background: #0f172a !important; font-family: 'Segoe UI', sans-serif; }
.tab-nav button { background: #1e293b !important; color: #94a3b8 !important; border-radius: 8px 8px 0 0 !important; }
.tab-nav button.selected { background: #6366f1 !important; color: white !important; }
h1 { text-align: center; color: #818cf8; font-size: 2rem; }
"""

with gr.Blocks(css=CSS, title="SentiViet Pro – BiLSTM+Attention") as demo:
    gr.Markdown("# 🧠 SentiViet Pro – BiLSTM + Attention\n**Phân tích cảm xúc văn bản tiếng Việt**")

    with gr.Tabs():
        # ── Tab 1: Huấn luyện ──────────────────────────────
        with gr.Tab("🏋️ Huấn luyện"):
            data_info_box = gr.Markdown(value=get_data_info)

            with gr.Row():
                train_btn    = gr.Button("🚀 Bắt đầu Huấn luyện", variant="primary", scale=2)
                status_label = gr.Textbox(label="Trạng thái", interactive=False, scale=3)

            with gr.Row():
                acc_plot  = gr.Plot(label="Accuracy theo Epoch")
                loss_plot = gr.Plot(label="Loss theo Epoch")

            summary_box = gr.Textbox(label="Model Summary", lines=20, interactive=False)
            report_box = gr.Textbox(label="Classification Report (4 chu so thap phan)", lines=10, interactive=False)

            train_btn.click(
                fn=build_and_train_model,
                outputs=[status_label, acc_plot, loss_plot, summary_box, report_box]
            )

        # -- Tab 2: So sanh mo hinh ------------------------------------------------
        with gr.Tab("So sanh mo hinh"):
            gr.Markdown(
                "### So sanh mo hinh de xuat voi cac mo hinh khac"
                "\nBang va bieu do dung cac chi so 4 chu so thap phan de de dua vao bao cao."
            )
            with gr.Row():
                compare_epochs = gr.Slider(minimum=3, maximum=30, value=8, step=1, label="So epoch toi da cho CNN/LSTM/BiLSTM baseline")
                include_phobert = gr.Checkbox(label="Chay them Transformer + PhoBERT (can transformers/torch va model vinai/phobert-base)", value=False)
            compare_btn = gr.Button("Chay so sanh truc tiep", variant="primary")
            compare_status = gr.Textbox(label="Trang thai so sanh", lines=4, interactive=False)
            compare_table = gr.Dataframe(label="Bang thong so so sanh", interactive=False, wrap=True)
            compare_chart = gr.Plot(label="Bieu do so sanh Accuracy / Macro F1 / Weighted F1")
            compare_report = gr.Textbox(label="Classification report tung mo hinh (4 chu so thap phan)", lines=24, interactive=False)

            compare_btn.click(fn=run_model_comparison, inputs=[compare_epochs, include_phobert], outputs=[compare_status, compare_table, compare_chart, compare_report])

        # ── Tab 2: Dự đoán ─────────────────────────────────
        with gr.Tab("🔍 Dự đoán"):
            with gr.Row():
                input_text  = gr.Textbox(label="Nhập văn bản cần phân tích",
                                         placeholder="Ví dụ: Thầy dạy rất hay và dễ hiểu...",
                                         lines=3, scale=3)
                predict_btn = gr.Button("🔮 Phân tích", variant="primary", scale=1)

            result_html = gr.HTML()
            prob_chart  = gr.Plot(label="Phân phối xác suất")

            with gr.Row():
                neg_pct = gr.Textbox(label="😡 Tiêu cực", interactive=False)
                neu_pct = gr.Textbox(label="😐 Trung tính", interactive=False)
                pos_pct = gr.Textbox(label="😍 Tích cực",  interactive=False)

            predict_btn.click(
                fn=predict_sentiment,
                inputs=input_text,
                outputs=[result_html, prob_chart, neg_pct, neu_pct, pos_pct]
            )

        # ── Tab 3: Thêm dữ liệu ────────────────────────────
        with gr.Tab("➕ Thêm dữ liệu"):
            with gr.Row():
                new_text  = gr.Textbox(label="Câu văn mới", lines=2, scale=3)
                new_label = gr.Dropdown(
                    choices=[("😡 Tiêu cực", 0), ("😐 Trung tính", 1), ("😍 Tích cực", 2)],
                    label="Nhãn", value=2, scale=1
                )
            add_btn    = gr.Button("➕ Thêm vào tập train", variant="secondary")
            add_status = gr.Textbox(label="Kết quả", interactive=False)
            data_info2 = gr.Markdown()

            add_btn.click(
                fn=add_single_sentence,
                inputs=[new_text, new_label],
                outputs=[add_status, data_info2]
            )

            gr.Markdown("---")
            gr.Markdown(
                "### 📂 Chế độ 2: Gộp thêm bộ dữ liệu CSV vào dataset hiện tại\n"
                "Bắt buộc tải đủ **2 file**: một file **Train** và một file **Val/Test**. "
                "File nên có cột `sentence` và `sentiment`. Nhãn hợp lệ: "
                "`negative`, `neutral`, `positive` hoặc `0`, `1`, `2`. "
                "Sau khi gộp xong, model sẽ được reset và cần train lại từ đầu."
            )

            with gr.Row():
                train_csv_file = gr.File(
                    label="File Train CSV để gộp",
                    file_types=[".csv"],
                    type="filepath"
                )
                val_csv_file = gr.File(
                    label="File Val/Test CSV để gộp",
                    file_types=[".csv"],
                    type="filepath"
                )

            remove_dup_checkbox = gr.Checkbox(
                label="Bỏ câu trùng khi gộp dữ liệu",
                value=True
            )

            append_csv_btn = gr.Button("➕ Gộp 2 file CSV vào dataset hiện tại", variant="primary")
            csv_status = gr.Textbox(label="Trạng thái gộp CSV", lines=9, interactive=False)
            csv_data_info = gr.Markdown(value=get_data_info)

            append_csv_btn.click(
                fn=append_csv_pair,
                inputs=[train_csv_file, val_csv_file, remove_dup_checkbox],
                outputs=[csv_status, csv_data_info]
            )

demo.queue()
demo.launch(share=True, debug=True, show_error=True)


✅ GPU: 1 thiết bị (Mixed Float16 bật)
✅ Đã tải: 8144 câu train, 2036 câu val


/tmp/ipykernel_738/1457852143.py:1044: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="SentiViet Pro – BiLSTM+Attention") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f6f79daaa5bf790b5a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Epoch 1/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - accuracy: 0.3482 - loss: 1.4567 - val_accuracy: 0.3355 - val_loss: 1.1531 - learning_rate: 3.0000e-04
Epoch 2/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.3507 - loss: 1.2066 - val_accuracy: 0.3345 - val_loss: 1.1469 - learning_rate: 3.0000e-04
Epoch 3/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.3595 - loss: 1.1702 - val_accuracy: 0.4337 - val_loss: 1.1422 - learning_rate: 3.0000e-04
Epoch 4/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.3992 - loss: 1.1378 - val_accuracy: 0.6729 - val_loss: 0.9970 - learning_rate: 3.0000e-04
Epoch 5/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.5874 - loss: 0.9101 - val_accuracy: 0.6493 - val_loss: 0.7287 - learning_rate: 3.0000e-04
Epoch 6/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.6450 - loss: 0.7894 - val_accuracy: 0.6847 - val_loss: 0.7027 - learning_rate: 3.0000e-04
Epoch 7/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - ac

Epoch 1/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.3356 - loss: 1.4660 - val_accuracy: 0.3576 - val_loss: 1.1487 - learning_rate: 3.0000e-04
Epoch 2/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.3479 - loss: 1.2094 - val_accuracy: 0.3679 - val_loss: 1.1469 - learning_rate: 3.0000e-04
Epoch 3/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.3530 - loss: 1.1754 - val_accuracy: 0.3581 - val_loss: 1.1487 - learning_rate: 3.0000e-04
Epoch 4/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.3572 - loss: 1.1631 - val_accuracy: 0.4794 - val_loss: 1.1480 - learning_rate: 3.0000e-04
Epoch 5/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.3667 - loss: 1.1536 - val_accuracy: 0.5275 - val_loss: 1.1405 - learning_rate: 3.0000e-04
Epoch 6/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.4315 - loss: 1.1031 - val_accuracy: 0.6071 - val_loss: 0.9325 - learning_rate: 3.0000e-04
Epoch 7/80
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - acc

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> None
Killing tunnel 127.0.0.1:7861 <> https://f6f79daaa5bf790b5a.gradio.live
